Structure:

01_data_cleaning_eda.ipynb
        
cleaned_credit_card_data.csv
        
02_variable_analysis.ipynb
        
Which variables appear useful?
        
03_bootstrap_analysis.ipynb

04_decision_tree.ipynb

In [2]:
# Credit Card Default Variable Analysis

#This notebook examines which variables in the credit card dataset
#have statistically significant relationships with customer default.
#Statistical methods, including descriptive statistics,
#t-tests, chi-square tests, and p-values,
#are used to evaluate the variables before predictive modeling.

# Import libraries needed for data manipulation, statistical testing,
# numerical calculations, and visualization.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency
from scipy.stats import zscore

In [3]:
## 2. Load the Cleaned Dataset

#The cleaned credit card dataset is loaded for statistical analysis.
#This dataset was previously prepared and
#validated in the data cleaning notebook.

# Upload the cleaned credit card dataset to the Google Colab environment.
#File to upload name: cleaned_credit_card_data.csv
from google.colab import files
uploaded = files.upload()

Saving cleaned_credit_card_data.csv to cleaned_credit_card_data.csv


In [4]:
## 3. Load and Verify the Dataset

# Load the cleaned dataset into a pandas DataFrame.
df = pd.read_csv("cleaned_credit_card_data.csv")

# Display the first five observations to verify that
# the dataset loaded correctly.
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [5]:
## 4. Verify the Dataset

# Verify the number of observations and variables in the dataset.
print("Number of Customer Observations:", df.shape[0])
print("Number of Variables:", df.shape[1])

# Verify that the cleaned dataset contains no missing values.
print("Total Missing Values:", df.isnull().sum().sum())

# Verify that the target variable contains only the expected
# binary values of 0 (no default) and 1 (default).
target_values = sorted(df["DEFAULT"].unique().tolist())
print("Target Variable Values:", target_values)

Number of Customer Observations: 30000
Number of Variables: 24
Total Missing Values: 0
Target Variable Values: [0, 1]


In [6]:
## 5. Separate Variables by Type

# Numerical variables contain measured amounts or continuous values.
# These variables will be evaluated using descriptive statistics
# and independent-samples t-tests.

numerical_variables = [
    "LIMIT_BAL",
    "AGE",
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6"
]

# Categorical variables represent groups rather than measured amounts.
# These variables will be evaluated using chi-square tests.

categorical_variables = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

print("Number of Numerical Variables:", len(numerical_variables))
print("Number of Categorical Variables:", len(categorical_variables))

Number of Numerical Variables: 14
Number of Categorical Variables: 9


In [9]:
## 6. T-Tests for Numerical Variables

# Create separate groups for customers who did and did not default.
no_default = df[df["DEFAULT"] == 0]
default = df[df["DEFAULT"] == 1]

# Create an empty list to store the results from each t-test.
t_test_results = []

# Perform an independent-samples t-test for each numerical variable.
for variable in numerical_variables:

    t_statistic, p_value = ttest_ind(
        no_default[variable],
        default[variable],
        equal_var=False
    )

    # Format very small p-values so they are easier to interpret.
    if p_value < 0.0001:
        formatted_p_value = "< 0.0001"
    else:
        formatted_p_value = f"{p_value:.4f}"

    # Store the formatted means and statistical test results.
    t_test_results.append({
        "Variable": variable,
        "No Default Mean": f"{no_default[variable].mean():.2f}",
        "Default Mean": f"{default[variable].mean():.2f}",
        "T-Statistic": f"{t_statistic:.2f}",
        "P-Value": formatted_p_value
    })

# Convert the results into a DataFrame.
t_test_summary = pd.DataFrame(t_test_results)

t_test_summary

,Variable,No Default Mean,Default Mean,T-Statistic,P-Value
0,LIMIT_BAL,178099.73,130109.66,28.95,< 0.0001
1,AGE,35.42,35.73,-2.32,0.0204
2,BILL_AMT1,51994.23,48509.16,3.40,0.0007
3,BILL_AMT2,49717.44,47283.62,2.45,0.0144
4,BILL_AMT3,47533.37,45181.60,2.46,0.0139
5,BILL_AMT4,43611.17,42036.95,1.76,0.0786
6,BILL_AMT5,40530.45,39540.19,1.16,0.2451
7,BILL_AMT6,39042.27,38271.44,0.93,0.3523
8,PAY_AMT1,6307.34,3397.04,17.51,< 0.0001
9,PAY_AMT2,6640.47,3388.65,14.82,< 0.0001


Explanation:

The t-test results compare the average value of each numerical variable between customers who defaulted and those who did not.

Variables with a p-value below 0.05 show a statistically significant difference between the two groups. These results can help us identify which of the numerical variables may be useful for predicting future customer defaults.

In [8]:
## 7. Chi-Square Tests for Categorical Variables

# Create an empty list to store the results of each chi-square test.
chi_square_results = []

# Test whether each categorical variable is associated with
# the customer's default status.
for variable in categorical_variables:

    # Create a contingency table comparing each category
    # of the variable with the customer's default status.
    contingency_table = pd.crosstab(
        df[variable],
        df["DEFAULT"]
    )

    # Perform the chi-square test of independence.
    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )

    # Format very small p-values so they are easier to interpret.
    if p_value < 0.0001:
        formatted_p_value = "< 0.0001"
    else:
        formatted_p_value = f"{p_value:.4f}"

    # Store the formatted results.
    chi_square_results.append({
        "Variable": variable,
        "Chi-Square": f"{chi2:.2f}",
        "Degrees of Freedom": dof,
        "P-Value": formatted_p_value
    })

# Convert the results into a DataFrame.
chi_square_summary = pd.DataFrame(chi_square_results)

chi_square_summary

,Variable,Chi-Square,Degrees of Freedom,P-Value
0,SEX,47.71,1,< 0.0001
1,EDUCATION,163.22,6,< 0.0001
2,MARRIAGE,35.66,3,< 0.0001
3,PAY_0,5365.96,10,< 0.0001
4,PAY_2,3474.47,10,< 0.0001
5,PAY_3,2622.46,10,< 0.0001
6,PAY_4,2341.47,10,< 0.0001
7,PAY_5,2197.69,9,< 0.0001
8,PAY_6,1886.84,9,< 0.0001


Explanation:

The chi-square tests examine whether each of the categorical variables has a statistically significant connection with a customer’s default status.

All the variables tested had a p-value below 0.05, which shows a significant association with default status. Meaning from these results, it can help us identify which categorical variables may be useful for predicting future customer defaults.

In [12]:
## 8. Calculate Cramer's V Effect Size

# Calculate Cramer's V for each categorical variable.
# Cramer's V measures the strength of the association between
# a categorical predictor and the DEFAULT target variable.

cramers_v_results = []

for variable in categorical_variables:

    # Create the contingency table.
    contingency_table = pd.crosstab(
        df[variable],
        df["DEFAULT"]
    )

    # Perform the chi-square test.
    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )

    # Determine the number of observations and table dimensions.
    n = contingency_table.to_numpy().sum()
    rows, columns = contingency_table.shape

    # Calculate Cramer's V.
    cramers_v = np.sqrt(
        chi2 / (n * min(rows - 1, columns - 1))
    )

    # Store the formatted results.
    cramers_v_results.append({
        "Variable": variable,
        "Cramer's V": f"{cramers_v:.3f}"
    })

# Convert the results into a DataFrame.
cramers_v_summary = pd.DataFrame(cramers_v_results)

# Sort from strongest to weakest observed association.
cramers_v_summary = cramers_v_summary.sort_values(
    by="Cramer's V",
    ascending=False
)

cramers_v_summary

,Variable,Cramer's V
3,PAY_0,0.423
4,PAY_2,0.340
5,PAY_3,0.296
6,PAY_4,0.279
7,PAY_5,0.271
8,PAY_6,0.251
1,EDUCATION,0.074
0,SEX,0.040
2,MARRIAGE,0.034


## 9. Interpretation of Statistical Tests

The statistical tests show that several variables have a significant relationship with credit card default. The t-tests found statistically significant differences between the default and non-default groups for LIMIT_BAL, AGE, BILL_AMT1 through BILL_AMT3, and PAY_AMT1 through PAY_AMT6 at a significance level of 0.05. BILL_AMT4 through BILL_AMT6 did not show statistically significant differences between the two groups.

The chi-square tests found statistically significant associations between all of the categorical variables and default status. However, the Cramer's V results show that the repayment status variables have a much stronger association with default than the demographic variables. PAY_0 had the strongest association, followed by PAY_2 through PAY_6.

These results suggest that repayment history, credit limit, and previous payment amounts may be useful variables to examine when developing the predictive models. However, statistical significance alone does not determine how useful a variable will be for prediction.


##10. Strongest Observed Variables

1.)PAY_0 — Cramér’s V = 0.423

2.)PAY_2 — 0.340

3.)PAY_3 — 0.296

4.)PAY_4 - 0.279

5.)PAY_5 — 0.271

6.)PAY_6 — 0.251

7.)EDUCATION — 0.074

8.)SEX — 0.040

9.)MARRIAGE — 0.034

Numerical Variables:

The t-test results showed that LIMIT_BAL and the PAY_AMT variables had some of the strongest statistically significant differences between customers who defaulted and those who did not. This suggests that credit limit and previous payment amounts may be useful variables when predicting credit card default.

##11. Strongest Numerical Variables

1. LIMIT_BAL — |t| = 28.95

2. PAY_AMT1 — 17.51

3. PAY_AMT2 — 14.82

4. PAY_AMT4 — 12.22

5. PAY_AMT3 — 11.89

6. PAY_AMT5 — 11.25

7. PAY_AMT6 — 11.06

8. BILL_AMT1 — 3.40

9. BILL_AMT3 — 2.46

10. BILL_AMT2 — 2.45

11. AGE — 2.32

12. BILL_AMT4 — 1.76, not significant at α = .05

13. BILL_AMT5 — 1.16, not significant

14. BILL_AMT6 — 0.93, not significant

Categorical Variables:

The chi-square and Cramér’s V results showed that the PAY repayment status variables had the strongest associations with default. PAY_0 had the strongest association, followed by PAY_2 through PAY_6. This suggests that a customer’s recent repayment history may be particularly useful for predicting default.

## 10. Conclusion

The statistical analysis identified several variables that may be useful for understanding credit card default. Repayment status variables, particularly PAY_0 through PAY_6, showed stronger associations with default than the demographic categorical variables. LIMIT_BAL and the previous payment amount variables also showed statistically significant differences between customers who defaulted and those who did not.

These results will be used as a reference during the modeling stage of the project. The Decision Tree model will evaluate the predictive usefulness of the variables when they are considered together. Bootstrap analysis will be used to evaluate the variability and uncertainty of estimates obtained from the dataset.